# gramlab on Kaggle — compact LM: prepare → train → fine-tune → chat

Settings needed (right sidebar): **Accelerator: GPU T4 x2 or P100**, **Internet: On**. Run cells top to bottom.


In [ ]:
import os, subprocess, sys
os.chdir('/kaggle/working')
if not os.path.exists('gramlab'):
    subprocess.run(['git','clone','--depth','1','https://github.com/furevych/gramlab.git'], check=True)
os.chdir('/kaggle/working/gramlab')
os.makedirs('external', exist_ok=True)
for url, d in [('https://github.com/gucorpling/amalgum','external/amalgum'),
               ('https://github.com/UniversalDependencies/UD_English-GUM','external/UD_English-GUM')]:
    if not os.path.exists(d):
        subprocess.run(['git','clone','--depth','1',url,d], check=True, env={**os.environ,'GIT_LFS_SKIP_SMUDGE':'1'})
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
import tokenizers; print('tokenizers', tokenizers.__version__)
print(os.getcwd(), os.listdir('.'))

## 1. Tokenizer + token shards (≈1–2 min)
BPE 16k by default. If the BPE path errors, rerun with `--set tokenizer.type=word` (then also pass it to every later command).

In [ ]:
!python -m gramlab.prepare --config configs/tiny.json

## 2. Pretrain the tiny model (≈5–15 min on a T4)
`eval_bits_per_byte` is the number to watch; perplexity is on gold GUM text the model never sees in training.

In [ ]:
!python -m gramlab.train --config configs/tiny.json

## 3. Fine-tune on synthetic Q&A so it answers in turns (≈1 min)

In [ ]:
!python scripts/make_wiki_qa.py --amalgum external/amalgum/amalgum --out data/wiki_qa.jsonl
!python -m gramlab.finetune --init_from work/runs/tiny/ckpt.pt --corpus wiki_qa --out tiny_sft --epochs 3

## 4. Evaluate

In [ ]:
!python -m gramlab.evaluate --ckpt work/runs/tiny/ckpt.pt --corpus gum_test

## 5. Chat
Type in the box and press **Send** (or call `ask('...')`). `MODE='base'` continues your text with the pretrained model; `MODE='chat'` uses the fine-tuned one in user/assistant turns.

Expectation: right *shape* of an answer, invented facts — it's a 3M-parameter model trained on 3.5M tokens.

In [ ]:
import sys, torch
sys.path.insert(0, '/kaggle/working/gramlab')
from gramlab.ckpt import load_checkpoint
from gramlab.tokenizer import USER, ASSISTANT, END, EOS
MODE = 'chat'   # 'chat' (fine-tuned) or 'base' (pretrained, continues your text)
CKPT = 'work/runs/tiny_sft/ckpt.pt' if MODE == 'chat' else 'work/runs/tiny/ckpt.pt'
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
model, tok, state = load_checkpoint(CKPT, dev)
history = []
def ask(text, temperature=0.8, top_k=40, top_p=0.95, rep=1.2, max_new=80):
    global history
    if MODE == 'chat':
        history.append(('user', text))
        ids = []
        for role, t in history[-6:]:
            ids += [USER if role == 'user' else ASSISTANT] + tok.encode(t) + [END]
        ids += [ASSISTANT]; stop = (END, EOS)
    else:
        ids = tok.encode(text); stop = ()
    x = torch.tensor([ids[-(model.ctx - 1):]], dtype=torch.long, device=dev)
    out = model.generate(x, max_new_tokens=max_new, temperature=temperature, top_k=top_k, top_p=top_p, repetition_penalty=rep, stop_ids=stop)
    reply = tok.decode(out[0, x.size(1):].tolist())
    if MODE == 'chat':
        history.append(('assistant', reply))
    return reply
print(ask('Tell me about Gothenburg.'))

In [ ]:
import ipywidgets as w
from IPython.display import display
box = w.Text(placeholder='say something...', layout=w.Layout(width='70%'), continuous_update=False)
btn = w.Button(description='Send'); out = w.Output()
def on_send(_=None):
    text = box.value.strip()
    if not text: return
    box.value = ''
    with out:
        print('you>   ', text); print('model> ', ask(text)); print()
btn.on_click(on_send)
box.observe(lambda ch: on_send() if ch['name'] == 'value' and ch['new'] else None, names='value')
display(w.HBox([box, btn]), out)

## 6. Next experiments
- Grammar heads arm: `!python -m gramlab.train --config configs/tiny_upos.json` then evaluate both.
- Continue on new data: add a corpus to `corpora.json`, `prepare --corpus <name>`, then `train --config configs/small.json --init_from work/runs/tiny/ckpt.pt`.
- Distill: `--set train.distill.teacher=work/runs/<teacher>/ckpt.pt`.
- Save a checkpoint out of Kaggle: right sidebar → Output → `work/runs/tiny_sft/ckpt.pt`.